## Pandas analysis

This exercise consists in analyzing a dataset containg timing information from a series of Time-to-Digital-Converters (TDC) implemented in a couple of FPGAs. Each measurement (i.e. each row of the input file) consists of a flag that specifies the type of message ('HEAD', which in this case is always 1), two addresses of the TDC providing the signal ('FPGA' and 'TDC_CHANNEL'), and the timing information ('ORBIT_CNT', 'BX_COUNTER', and 'TDC_MEAS'). Each TDC count corresponds to 25/30 ns, whereas a unit of BX_COUNTER corresponds to 25 ns, and the ORBIT_CNT is increased every 'x' BX_COUNTER. This allows to store the time in a similar way to hours, minutes and seconds.

In [ ]:
# If you didn't download it yet, please get the relevant file now!
#!wget https://www.dropbox.com/s/xvjzaxzz3ysphme/data_000637.txt -P ~/data/

1\. Create a Pandas DataFrame reading N rows of the 'data_000637.txt' dataset. Choose N to be smaller than or equal to the maximum number of rows and larger that 10k.

2\. Find out the number of BX in a ORBIT (the value 'x').

3\. Find out how much the data taking lasted. You can either make an estimate based on the fraction of the measurements (rows) you read, or perform this check precisely by reading out the whole dataset.

4\. Create a new column with the absolute time in ns (as a combination of the other three columns with timing information).

5\. Replace the values (all 1) of the HEAD column randomly with 0 or 1.

6\. Create a new DataFrame that contains only the rows with HEAD=1.

7\. Make two occupancy plots (one for each FPGA), i.e. plot the number of counts per TDC channel

8\. Use the groupby method to find out the noisy channels, i.e. the TDC channels with most counts (say the top 3)

9\. Count the number of unique orbits. Count the number of unique orbits with at least one measurement from TDC_CHANNEL=139

### Libraries

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [24]:
# 1. Read 10k < N < Maxrows of the dEachata file.
path = '~/data/data_000637.txt'
N_rows = 10001
df = pd.read_csv(path,nrows=N_rows)
df.head()

,HEAD,FPGA,TDC_CHANNEL,ORBIT_CNT,BX_COUNTER,TDC_MEAS
0,1,0,123,3869200167,2374,26
1,1,0,124,3869200167,2374,27
2,1,0,63,3869200167,2553,28
3,1,0,64,3869200167,2558,19
4,1,0,64,3869200167,2760,25


In [25]:
df.describe()

,HEAD,FPGA,TDC_CHANNEL,ORBIT_CNT,BX_COUNTER,TDC_MEAS
count,10001.0,10001.00000,10001.000000,1.000100e+04,10001.000000,10001.000000
mean,1.0,0.29847,55.447055,3.869200e+09,1811.483752,13.378762
std,0.0,0.45761,39.416307,2.878557e+01,1064.245349,9.223759
min,1.0,0.00000,1.000000,3.869200e+09,0.000000,0.000000
25%,1.0,0.00000,24.000000,3.869200e+09,835.000000,5.000000
50%,1.0,0.00000,56.000000,3.869200e+09,1812.000000,13.000000
75%,1.0,1.00000,64.000000,3.869200e+09,2782.000000,21.000000
max,1.0,1.00000,139.000000,3.869200e+09,3563.000000,29.000000


In [26]:
# 2. Number of BX in an ORBIT (value 'x')
orbit_bx_num = np.array(df.groupby('ORBIT_CNT')['BX_COUNTER'].sum())
print(len(orbit_bx_num))
print(orbit_bx_num)

101
[124133  97201 144343 217462 228822 162033 153284 211070 312015 148031
 169391  64134 191251 263233 264036  53081 157916 136845 217906 129093
 186390 138524  96363 178784 246785 172087 281178 262861 247526  55822
 149723 234860 347968 119238  87398 192469 205196 189218 181098  92747
 244849 290110 151416  66084 218861 190490 200464  94649 222372 245760
  79613 343071  56277 151294 154943 256551  94540 182810 245297  91548
 218937 175361 162758 264100 208633 147614 107674 201718 285568 110368
 211372 148063 282905 192344 272065 170675 103882 185032 282000 214690
  35001 133860 267807 223546  36784 209027 132359 292222 152105 125240
 197546 215393 141225 112020 213609 225506 279850 158363 148300 166816
  11797]


In [27]:
df.groupby('ORBIT_CNT').sum()

,HEAD,FPGA,TDC_CHANNEL,BX_COUNTER,TDC_MEAS
ORBIT_CNT,,,,,
3869200167,43,6,2683,124133,619
3869200168,85,36,4376,97201,1134
3869200169,127,37,6727,144343,1688
3869200170,98,32,5183,217462,1308
3869200171,109,30,6562,228822,1482
3869200172,89,26,4813,162033,1154
3869200173,88,26,5487,153284,1194
3869200174,128,50,6368,211070,1570
3869200175,128,39,6369,312015,1771


In [28]:
# 3. How much the data taking lasted. 
# You can either make an estimate based on the fraction of the measurements 
# (rows) you read, or perform this check precisely by reading out the whole dataset.
# -------------------------------------------------------------------------------------
# | Each TDC count corresponds to 25/30 ns, a unit of BX_COUNTER corresponds to 25 ns,|
# | and the ORBIT_CNT is increased every 'x' BX_COUNTER                               |
# -------------------------------------------------------------------------------------

time_estimate = df['TDC_MEAS']*(25/30) + df['BX_COUNTER']*25
time_estimate

0        59371.666667
1        59372.500000
2        63848.333333
3        63965.833333
4        69020.833333
5        69053.333333
6        69311.666667
7        69400.000000
8        69367.500000
9        69705.833333
10       69628.333333
11       69665.833333
12       69815.000000
13       69794.166667
14       69727.500000
15       69925.000000
16       69686.666667
17       69758.333333
18       69878.333333
19       69921.666667
20       69736.666667
21       69733.333333
22       69769.166667
23       69987.500000
24       69890.833333
25       69936.666667
26       69986.666667
27       77042.500000
28       77025.833333
29       77091.666667
             ...     
9971     46119.166667
9972     46014.166667
9973     46225.833333
9974     46169.166667
9975     46179.166667
9976     46117.500000
9977     46463.333333
9978     46128.333333
9979     46270.000000
9980     46485.833333
9981     46206.666667
9982     46284.166667
9983     15202.500000
9984     15140.000000
9985     1

In [40]:
# 4. New column with the absolute time in ns (as a combination of the other three columns with timing information).
df_copy = df.copy()
df_copy['absolute_time_ns'] = df_copy['TDC_MEAS']*25/30 + df_copy['BX_COUNTER']*25
df_copy.head()

,HEAD,FPGA,TDC_CHANNEL,ORBIT_CNT,BX_COUNTER,TDC_MEAS,absolute_time_ns
0,1,0,123,3869200167,2374,26,59371.666667
1,1,0,124,3869200167,2374,27,59372.500000
2,1,0,63,3869200167,2553,28,63848.333333
3,1,0,64,3869200167,2558,19,63965.833333
4,1,0,64,3869200167,2760,25,69020.833333


In [79]:
# 5. Replace the values (all 1) of the HEAD column randomly with 0 or 1.

df_copy['HEAD'] = np.random.randint(0,2,size=N_rows)
df_copy.describe()

,HEAD,FPGA,TDC_CHANNEL,ORBIT_CNT,BX_COUNTER,TDC_MEAS,absolute_time_ns
count,10001.000000,10001.00000,10001.000000,1.000100e+04,10001.000000,10001.000000,10001.000000
mean,0.501250,0.29847,55.447055,3.869200e+09,1811.483752,13.378762,45298.242759
std,0.500023,0.45761,39.416307,2.878557e+01,1064.245349,9.223759,26606.206529
min,0.000000,0.00000,1.000000,3.869200e+09,0.000000,0.000000,0.833333
25%,0.000000,0.00000,24.000000,3.869200e+09,835.000000,5.000000,20878.333333
50%,1.000000,0.00000,56.000000,3.869200e+09,1812.000000,13.000000,45300.000000
75%,1.000000,1.00000,64.000000,3.869200e+09,2782.000000,21.000000,69566.666667
max,1.000000,1.00000,139.000000,3.869200e+09,3563.000000,29.000000,89099.166667


In [90]:
# 6. New DataFrame that contains only the rows with HEAD=1.
df_head_one = df_copy[df_copy['HEAD']>0]
df_head_one.describe()

,HEAD,FPGA,TDC_CHANNEL,ORBIT_CNT,BX_COUNTER,TDC_MEAS,absolute_time_ns
count,5013.0,5013.000000,5013.000000,5.013000e+03,5013.000000,5013.000000,5013.000000
mean,1.0,0.295631,55.466786,3.869200e+09,1812.065430,13.417115,45312.816677
std,0.0,0.456372,39.609425,2.881980e+01,1072.547904,9.243026,26813.865360
min,1.0,0.000000,1.000000,3.869200e+09,0.000000,0.000000,15.833333
25%,1.0,0.000000,24.000000,3.869200e+09,832.000000,5.000000,20800.000000
50%,1.0,0.000000,56.000000,3.869200e+09,1817.000000,13.000000,45440.000000
75%,1.0,1.000000,64.000000,3.869200e+09,2790.000000,21.000000,69758.333333
max,1.0,1.000000,139.000000,3.869200e+09,3563.000000,29.000000,89099.166667


In [93]:
# 7. Make two occupancy plots (one for each FPGA), i.e. plot the number of counts per TDC channel
df_fpga = df_head_one.groupby('FPGA')


BX_COUNTER                                                                \
          count         mean          std  min    25%     50%     75%     max   
FPGA                                                                            
0        7016.0  1810.264681  1058.851345  0.0  849.0  1810.0  2763.0  3563.0   
1        2985.0  1814.349079  1076.990333  1.0  819.0  1817.0  2800.0  3563.0   

        HEAD       ...  TDC_CHANNEL        TDC_MEAS                            \
       count mean  ...          75%    max    count       mean       std  min   
FPGA               ...                                                          
0     7016.0  1.0  ...         64.0  139.0   7016.0  13.388540  9.222233  0.0   
1     2985.0  1.0  ...         30.0  139.0   2985.0  13.355779  9.228848  0.0   

                             
      25%   50%   75%   max  
FPGA                         
0     5.0  13.0  21.0  29.0  
1     5.0  13.0  21.0  29.0  

[2 rows x 40 columns]